In [1]:
# This part is for 

from pathlib import Path
path_spec_data=Path.cwd().parent.parent/"spec_data"
path_benchmark_data=Path.cwd().parent.parent/"benchmark_for_test"

path_spec_data.mkdir(parents=True, exist_ok=True)
path_benchmark_data.mkdir(parents=True, exist_ok=True)

In [ ]:
cache_list_threshold=1_000_000

ion_mode=[1, -1]

num_per_group=[1_000_000,3_000_000, 10_000_000, 30_000_000, 100_000_000, 300_000_000]

dynamic_script_path="15-1_dynamic_entropy_search_group_size_hybrid_fast_update_mode_every_step_convert_to_flash.py"

In [3]:
import msgpack
import subprocess
import os
import shutil
from typing import Union

def run_usrbintime_by_arguments(
          arguments:list[str], 
          if_output:bool=False, 
          output_memory_file:Union[str,Path]=None, 
          output_time_file:Union[str, Path]=None):
    
    # arguments: script_path, str(charge), step
    command=["/usr/bin/time","-v","python"] + arguments

    if if_output: # Output to files as record
        with open(output_memory_file, "w") as f1, open(output_time_file, "w") as f2:
            subprocess.run(command, stderr=f1, stdout=f2, cwd=Path.cwd(), env=os.environ.copy())

    else: # Output is not needed
         
        subprocess.run(command, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, cwd=Path.cwd(), env=os.environ.copy())
        
    return

for charge in ion_mode:
    for group_size in num_per_group:
        # remove the old index
        path_comparison_dynamic_data=Path.cwd().parent.parent/f"comparison_data/dynamic/charge-{charge}"
        if path_comparison_dynamic_data.exists():
            shutil.rmtree(path_comparison_dynamic_data)

        arguments=[dynamic_script_path, str(charge), str(group_size), str(cache_list_threshold)]
        output_memory_file=path_benchmark_data/f"dynamic_fast_update_{charge}_memory_usage_options_{group_size}_num_per_group.txt"
        output_time_file=path_benchmark_data/f"dynamic_fast_update_{charge}_compare_time_options_{group_size}_num_per_group.txt"
        run_usrbintime_by_arguments(arguments=arguments,
                                    if_output=True,
                                    output_memory_file=output_memory_file,
                                    output_time_file=output_time_file)

        